# 01 — EDA e pré-processamento

**Projeto Respira** — estimativa de nível de ansiedade a partir de hábitos e sinais vitais
Entrega 2 · Machine Learning · Projeto Final · *Dados prontos*

Ordem do notebook:

1. setup e carga;
2. `limpar(df)` — só operações linha a linha, **antes** do split;
3. split 80/20 com `random_state=42` (o mesmo da E1) — o teste fica trancado;
4. EDA com as cinco perguntas, só em `X_tr` e `y_tr`, cada gráfico com **Observação** e **Decisão**;
5. pré-processador (`ColumnTransformer`);
6. teste de sanidade (sem NaN, cross-val, comparação com baseline);
7. tabela de decisões.

> Regra: tudo que for aprendido dos dados acontece só com o treino. `X_te` e `y_te` não aparecem em nenhuma decisão.

---
## 1. Setup e carga

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 170)

SEED = 42
ALVO = "Anxiety Level (1-10)"

NOME = "enhanced_anxiety_dataset.csv"
CANDIDATOS = [f"../data/raw/{NOME}", f"data/raw/{NOME}", NOME, f"/content/{NOME}"]
CAMINHO = next((c for c in CANDIDATOS if os.path.exists(c)), None)
if CAMINHO is None:
    try:
        from google.colab import files
        print(f"Selecione o arquivo {NOME}:")
        CAMINHO = list(files.upload().keys())[0]
    except ImportError:
        raise FileNotFoundError(f"{NOME} nao encontrado. Coloque em data/raw/.")

print("Lendo:", CAMINHO)
df_bruto = pd.read_csv(CAMINHO)
print("Shape bruto:", df_bruto.shape)

---
## 2. `limpar(df)` — antes do split

Só entra aqui o que **não precisa olhar as outras linhas** (pergunta-teste da aula). Isso inclui:

- remover duplicatas exatas e colunas de ID (o dataset não tem ID);
- padronizar texto: `"yes "`, `"YES"`, `"Y"`, `"sim"` → `"Yes"`;
- ausentes disfarçados (`"?"`, `"NA"`, `""`) → `NaN`;
- limites de **domínio** (idade entre 0 e 120, sono entre 0 e 24 h, escalas 1–10…) → fora do limite vira `NaN` e o imputer do pipeline resolve. Isso também cobre sentinelas numéricas como `-1` e `999`, sem tocar no `0`, que é valor válido em álcool, cafeína e terapia.

Esta mesma função vai rodar na API (E4), antes do pipeline. Ela **não mexe no alvo**.

In [ ]:
COLS_BINARIAS = ["Smoking", "Family History of Anxiety", "Dizziness",
                 "Medication", "Recent Major Life Event"]
COLS_NOMINAIS = ["Gender", "Occupation"]

# limites de dominio (min, max) -- nao aprendem nada dos dados
LIMITES = {
    "Age": (0, 120),
    "Sleep Hours": (0, 24),
    "Physical Activity (hrs/week)": (0, 168),
    "Caffeine Intake (mg/day)": (0, 2000),
    "Alcohol Consumption (drinks/week)": (0, 200),
    "Stress Level (1-10)": (1, 10),
    "Heart Rate (bpm)": (30, 220),
    "Breathing Rate (breaths/min)": (4, 60),
    "Sweating Level (1-5)": (1, 5),
    "Therapy Sessions (per month)": (0, 31),
    "Diet Quality (1-10)": (1, 10),
}
COLS_NUMERICAS = list(LIMITES)

AUSENTES_TEXTO = {"", "?", "na", "n/a", "nan", "none", "null", "-"}
SIM = {"yes", "y", "sim", "s", "true", "1"}
NAO = {"no", "n", "nao", "não", "false", "0"}

def _bin(v):
    if pd.isna(v): return np.nan
    t = str(v).strip().lower()
    if t in AUSENTES_TEXTO: return np.nan
    if t in SIM: return "Yes"
    if t in NAO: return "No"
    return np.nan  # valor desconhecido numa binaria: trata como ausente

def _txt(v):
    if pd.isna(v): return np.nan
    t = str(v).strip()
    return np.nan if t.lower() in AUSENTES_TEXTO else t.title()

def limpar(df, verbose=False):
    """Limpeza linha a linha. Nao aprende nada dos dados; roda igual no treino e na API."""
    d = df.copy()
    d.columns = [c.strip() for c in d.columns]
    d = d.drop_duplicates()
    for c in COLS_BINARIAS:
        if c in d: d[c] = d[c].map(_bin)
    for c in COLS_NOMINAIS:
        if c in d: d[c] = d[c].map(_txt)
    for c, (lo, hi) in LIMITES.items():
        if c in d:
            d[c] = pd.to_numeric(d[c], errors="coerce")
            fora = (d[c] < lo) | (d[c] > hi)
            if verbose and fora.sum():
                print(f"  {c}: {int(fora.sum())} valores fora de [{lo}, {hi}] -> NaN")
            d.loc[fora, c] = np.nan
    return d

df = limpar(df_bruto, verbose=True)
print("Linhas removidas pela limpeza:", len(df_bruto) - len(df))
print("Nulos criados pela limpeza (X):", int(df.drop(columns=[ALVO]).isna().sum().sum()))
assert df[ALVO].notna().all(), "alvo com NaN: decidir antes de seguir"
df.head()

---
## 3. Split (o mesmo da E1)

`test_size=0.2`, `random_state=42`. O alvo é contínuo, então não há `stratify` (isso vale para classificação). A partir daqui, a EDA só olha `X_tr` e `y_tr`.

In [ ]:
X = df.drop(columns=[ALVO])
y = df[ALVO]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Treino: {len(X_tr):,}   Teste (trancado): {len(X_te):,}")
dados = X_tr.assign(**{ALVO: y_tr})   # so treino, para a EDA

---
## 4. EDA — cinco perguntas, cada uma termina numa decisão

### Pergunta 1 — Como o alvo se distribui?

In [ ]:
print(y_tr.describe().round(2).to_string())
print()
cont = y_tr.round().value_counts().sort_index()
print((cont / len(y_tr) * 100).round(1).rename("% do treino").to_string())
alto = (y_tr >= 8).mean() * 100
print(f"\nNotas 8 a 10: {alto:.1f}% do treino")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
y_tr.plot(kind="hist", bins=30, ax=ax[0], color="#6A1B9A")
ax[0].set_title("Distribuicao do alvo (treino)")
y_tr.plot(kind="box", vert=False, ax=ax[1], color="#6A1B9A")
ax[1].set_title("Boxplot do alvo")
plt.tight_layout(); plt.show()

**Observação:** o alvo é contínuo (1 a 10) e as notas altas (8 a 10) são raras — a célula acima mostra a fatia exata no treino (~9% na E1).

**Decisão:** não há classes, então não há `stratify` nem `class_weight`. Mantém MAE como métrica principal. Registrado para a E3: o modelo tende a errar mais nas notas altas; avaliar `sample_weight` para elas.

### Pergunta 2 — Onde faltam dados?

In [ ]:
falt = X_tr.isna().mean().sort_values(ascending=False) * 100
print(falt.round(2).to_string())
print(f"\nTotal de nulos em X_tr: {int(X_tr.isna().sum().sum())}")

# ausentes disfarcados: sentinelas numericas que passariam pela limpeza?
for c in COLS_NUMERICAS:
    lo, hi = LIMITES[c]
    print(f"{c:36s} min={X_tr[c].min():>8.1f}  max={X_tr[c].max():>8.1f}   dominio=[{lo}, {hi}]")

**Observação:** nenhuma coluna tem valores faltantes e nenhum valor ficou fora dos limites de domínio (a limpeza reportou 0 conversões). Consistente com dado sintético (E1).

**Decisão:** nenhuma ação sobre os dados de treino. **Mas** o pipeline mantém `SimpleImputer` em todos os caminhos: o usuário da API vai deixar campo vazio, e o pipeline precisa saber o que fazer. Como o percentual de faltantes é 0% (< 5%), sem `add_indicator`. Sem `dropna()`.

### Pergunta 3 — Como cada variável se distribui?

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(13, 11))
for ax, c in zip(axes.ravel(), COLS_NUMERICAS):
    X_tr[c].plot(kind="hist", bins=25, ax=ax, color="#1565C0")
    ax.set_title(c, fontsize=9); ax.set_ylabel("")
for ax in axes.ravel()[len(COLS_NUMERICAS):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

print("Assimetria (skew) -- |skew| > 1 sugere testar log:")
sk = X_tr[COLS_NUMERICAS].skew().sort_values(key=abs, ascending=False)
print(sk.round(2).to_string())

fig, ax = plt.subplots(figsize=(12, 3.2))
Z = (X_tr[COLS_NUMERICAS] - X_tr[COLS_NUMERICAS].mean()) / X_tr[COLS_NUMERICAS].std()
Z.boxplot(ax=ax, vert=False, grid=False)
ax.set_title("Boxplots (padronizados, so para comparar caudas)")
plt.tight_layout(); plt.show()

**Observação:** as 11 colunas numéricas têm escalas muito diferentes (cafeína em centenas de mg, estresse de 1 a 10, idade em dezenas). Os histogramas e os valores de `skew` acima mostram se alguma tem cauda longa; valores extremos que aparecem nos boxplots estão dentro do domínio plausível, então são reais.

**Decisão:** `StandardScaler` em todas as numéricas (escalas diferentes). Outliers dentro do domínio: **mantidos**. Se alguma coluna mostrar |skew| > 1, testar log na E3; fora isso, nenhuma transformação.

In [ ]:
for c in COLS_BINARIAS + COLS_NOMINAIS:
    print(f"{c}: " + " | ".join(f"{k}: {v}" for k, v in X_tr[c].value_counts().items()))

**Observação (categóricas):** `Gender` tem 3 categorias, `Occupation` tem 13, e as 5 binárias são `Yes`/`No`. Nenhuma categoria rara a ponto de quebrar o encoder.

**Decisão:** `Gender` e `Occupation` → nominais, `OneHotEncoder(handle_unknown="ignore")`. As 5 binárias → `OneHotEncoder(drop="if_binary")`. Nenhuma ordinal verdadeira (`Sweating`, `Stress`, `Diet` são escalas numéricas e entram como numéricas). Nenhuma coluna de ID ou texto livre.

### Pergunta 4 — O que se relaciona com o alvo?

In [ ]:
corr = X_tr[COLS_NUMERICAS].corrwith(y_tr).sort_values(key=abs, ascending=False)
print(corr.round(3).to_string())

fig, ax = plt.subplots(figsize=(7, 3.8))
corr.sort_values().plot(kind="barh", ax=ax,
    color=["#1565C0" if v < 0 else "#C62828" for v in corr.sort_values()])
ax.axvline(0, color="black", lw=0.8); ax.set_title("Correlacao com o alvo (treino)")
plt.tight_layout(); plt.show()

cats = COLS_BINARIAS + ["Gender"]
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, c in zip(axes.ravel(), cats):
    dados.boxplot(column=ALVO, by=c, ax=ax, grid=False)
    ax.set_title(c, fontsize=9); ax.set_xlabel("")
plt.suptitle("Alvo por categoria"); plt.tight_layout(); plt.show()

print(dados.groupby("Occupation")[ALVO].mean().sort_values().round(2).to_string())

**Observação:** estresse, sessões de terapia, sono (negativa), cafeína e atividade física (negativa) são as mais ligadas ao alvo (ver ranking acima). Terapia e `Medication` aparecem ligadas a ansiedade **mais alta** — causalidade reversa: quem já é ansioso procura terapia e remédio.

**Decisão:** manter todas as 18 colunas no pré-processador. Nenhuma coluna só existe depois do alvo, então não há vazamento no sentido estrito. `Therapy Sessions` e `Medication` ficam **sinalizadas**: na E3 repetir o experimento sem elas (E1: MAE 0,818 → 0,828) e decidir; nunca interpretá-las como fatores de risco.

### Pergunta 5 — Há colunas redundantes?

In [ ]:
cm = X_tr[COLS_NUMERICAS].corr()
fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(cm, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(cm))); ax.set_yticks(range(len(cm)))
ax.set_xticklabels(cm.columns, rotation=90, fontsize=7); ax.set_yticklabels(cm.columns, fontsize=7)
plt.colorbar(im); ax.set_title("Correlacao entre entradas (treino)")
plt.tight_layout(); plt.show()

pares = [(a, b, cm.loc[a, b]) for i, a in enumerate(cm.columns) for b in cm.columns[i+1:]
         if abs(cm.loc[a, b]) > 0.9]
print("Pares com |r| > 0,9:", pares if pares else "nenhum")

**Observação:** o heatmap e a lista acima mostram os pares de entradas com |r| > 0,9. Para um dataset em que as variáveis foram geradas a partir do alvo, espera-se correlação fraca entre as entradas.

**Decisão:** se a lista acima vier vazia, **nenhuma coluna é removida por redundância**. Se aparecer algum par, remover uma das duas e registrar na tabela.

---
## 5. Pré-processador

Um caminho por tipo de coluna. Ainda sem `fit`: ele só é treinado **dentro** do `Pipeline`, no `X_tr`.

In [ ]:
num_cols = COLS_NUMERICAS
nom_cols = COLS_NOMINAIS
bin_cols = COLS_BINARIAS

# garante que toda coluna de X tem um caminho (remainder='drop' esconderia esquecimento)
assert set(num_cols + nom_cols + bin_cols) == set(X_tr.columns), \
    set(X_tr.columns) ^ set(num_cols + nom_cols + bin_cols)

num = Pipeline([("imp", SimpleImputer(strategy="median")),
                ("esc", StandardScaler())])
nom = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                ("ohe", OneHotEncoder(handle_unknown="ignore"))])
bina = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                 ("ohe", OneHotEncoder(drop="if_binary", handle_unknown="ignore"))])

pre = ColumnTransformer([("num", num, num_cols),
                         ("nom", nom, nom_cols),
                         ("bin", bina, bin_cols)], remainder="drop")
pre

---
## 6. Teste de sanidade

Quatro checagens: sem NaN depois do transform, mesmas linhas, roda em cross-validation sem erro de categoria, e fica **acima do baseline**. Tudo no treino (`cv=5`); o teste continua trancado.

Os baselines da E1 foram medidos no teste; aqui comparo na **mesma régua** (CV no treino), refazendo B0 e B2.

In [ ]:
Xt = pre.fit_transform(X_tr)
print("Shape:", Xt.shape, "| linhas iguais ao treino:", Xt.shape[0] == len(X_tr))
print("NaN depois do transform:", int(np.isnan(Xt).sum()))
print(pre.get_feature_names_out()[:10])

In [ ]:
scoring = {"MAE": "neg_mean_absolute_error",
           "RMSE": "neg_root_mean_squared_error",
           "R2": "r2"}

# B0: media | B2: linear so com estresse | modelo simples: linear completa
b2_pre = ColumnTransformer([("s", "passthrough", ["Stress Level (1-10)"])], remainder="drop")
modelos = {
    "B0 - media do treino":        DummyRegressor(strategy="mean"),
    "B2 - linear so com estresse": Pipeline([("pre", b2_pre), ("m", LinearRegression())]),
    "Pipeline - regressao linear": Pipeline([("pre", pre), ("modelo", LinearRegression())]),
}
linhas = {}
for nome, m in modelos.items():
    r = cross_validate(m, X_tr, y_tr, cv=5, scoring=scoring)
    linhas[nome] = {"MAE": -r["test_MAE"].mean(), "MAE (desvio)": r["test_MAE"].std(),
                    "RMSE": -r["test_RMSE"].mean(), "R2": r["test_R2"].mean()}
cv_tab = pd.DataFrame(linhas).T.round(3)
display(cv_tab)

mae_pipe = cv_tab.loc["Pipeline - regressao linear", "MAE"]
mae_b2 = cv_tab.loc["B2 - linear so com estresse", "MAE"]
print(f"\n>>> Pipeline MAE {mae_pipe:.3f} contra o melhor baseline (B2) {mae_b2:.3f}: "
      f"ganho de {mae_b2 - mae_pipe:.3f} ponto.")
assert mae_pipe < mae_b2, "pipeline nao bate o baseline"

### Robustez: uma linha de API

O que a E4 vai mandar: uma linha só, campo vazio, categoria nunca vista. O pipeline precisa responder sem quebrar (é o que `get_dummies` não faz).

In [ ]:
pipe = Pipeline([("pre", pre), ("modelo", LinearRegression())]).fit(X_tr, y_tr)

linha = X_tr.iloc[[0]].copy()
linha["Occupation"] = "Astronauta"      # categoria nunca vista
linha["Sleep Hours"] = np.nan           # campo vazio
linha["Smoking"] = np.nan               # binaria vazia
print("Previsao com campos vazios e categoria nova:",
      round(float(pipe.predict(limpar(linha))[0]), 2))

# entrada suja passando pela limpar()
suja = X_tr.iloc[[0]].copy()
suja["Age"] = 230; suja["Smoking"] = " yes "; suja["Gender"] = " female"
print(limpar(suja)[["Age", "Smoking", "Gender"]].to_string())
print("Previsao:", round(float(pipe.predict(limpar(suja))[0]), 2))

---
## 7. Tabela de decisões

In [ ]:
tabela = pd.DataFrame([
 ["Age", "numérica", "sem nulos; dentro de 0-120", "limite de domínio (>120 vira NaN); mediana + StandardScaler", "limpar + pipeline"],
 ["Gender", "nominal (3)", "Male, Female, Other", "padronizar texto; moda + one-hot, ignore", "limpar + pipeline"],
 ["Occupation", "nominal (13)", "13 profissões, sem categoria rara problemática", "padronizar texto; moda + one-hot, ignore", "limpar + pipeline"],
 ["Sleep Hours", "numérica", "r negativa com o alvo; dentro de 0-24", "limite de domínio; mediana + StandardScaler", "limpar + pipeline"],
 ["Physical Activity (hrs/week)", "numérica", "r negativa fraca", "limite de domínio; mediana + StandardScaler", "limpar + pipeline"],
 ["Caffeine Intake (mg/day)", "numérica", "escala em centenas; r positiva", "limite de domínio; mediana + StandardScaler", "limpar + pipeline"],
 ["Alcohol Consumption (drinks/week)", "numérica", "0 é valor válido", "limite de domínio; mediana + StandardScaler", "limpar + pipeline"],
 ["Smoking", "binária", "Yes/No", "padronizar para Yes/No; moda + one-hot (drop if_binary)", "limpar + pipeline"],
 ["Family History of Anxiety", "binária", "Yes/No", "padronizar para Yes/No; moda + one-hot (drop if_binary)", "limpar + pipeline"],
 ["Stress Level (1-10)", "numérica", "maior correlação com o alvo", "limite 1-10; mediana + StandardScaler", "limpar + pipeline"],
 ["Heart Rate (bpm)", "numérica", "sinal vital, medido em casa", "limite 30-220; mediana + StandardScaler", "limpar + pipeline"],
 ["Breathing Rate (breaths/min)", "numérica", "sinal vital, medido em casa", "limite 4-60; mediana + StandardScaler", "limpar + pipeline"],
 ["Sweating Level (1-5)", "numérica (escala)", "escala curta", "limite 1-5; mediana + StandardScaler", "limpar + pipeline"],
 ["Dizziness", "binária", "Yes/No", "padronizar para Yes/No; moda + one-hot (drop if_binary)", "limpar + pipeline"],
 ["Medication", "binária", "causalidade reversa (consequência da ansiedade)", "mantida e sinalizada; ablação na E3", "limpar + pipeline"],
 ["Therapy Sessions (per month)", "numérica", "r positiva forte; causalidade reversa", "mantida e sinalizada; ablação na E3", "limpar + pipeline"],
 ["Recent Major Life Event", "binária", "Yes/No", "padronizar para Yes/No; moda + one-hot (drop if_binary)", "limpar + pipeline"],
 ["Diet Quality (1-10)", "numérica", "escala 1-10", "limite 1-10; mediana + StandardScaler", "limpar + pipeline"],
 ["Anxiety Level (1-10)", "alvo", "contínuo; notas 8-10 raras", "não é transformado; sem stratify; atenção a notas altas na E3", "—"],
], columns=["Coluna", "Tipo", "Achado na EDA", "Decisão", "Onde"])
tabela

In [ ]:
try:
    print(tabela.to_markdown(index=False))   # copie e cole no README
except ImportError:
    print(tabela.to_csv(index=False))

---
## Resumo da Entrega 2

- `limpar(df)`: só linha a linha, aplicada antes do split, reaproveitável pela API.
- Split igual ao da E1; teste trancado e não usado em nenhuma decisão.
- Pré-processador `ColumnTransformer` com 3 caminhos (numérico, nominal, binário), `handle_unknown="ignore"` e imputer em todos.
- Teste de sanidade: sem NaN, roda em cross-val, e bate o melhor baseline (B2) na mesma régua.
- Para a E3: o mesmo `pre`, trocando só o último passo do `Pipeline`.